### 📦 Step 1: Install Dependencies
This cell installs all required Python libraries to run the local AI stack.
* **LangChain:** The framework for orchestrating the AI logic.
* **ChromaDB:** The vector database used to store the therapist's knowledge.
* **Flask:** The web framework serving the HTML/JS frontend.
* **Sentence-Transformers:** Used to convert text into mathematical vectors (embeddings).
* **TF-Keras:** Required backend for some transformer operations.

*Note: The `!pip install` command runs a shell command from within the notebook.*

In [1]:
# Cell 1: Install Dependencies
# We install 'langchain' (main), 'langchain-chroma' (database connector),
# and 'langchain-huggingface' (embeddings) explicitly to avoid missing module errors.
!pip install langchain langchain-community langchain-core langchain-chroma langchain-huggingface chromadb flask sentence-transformers pypdf tf-keras

### ⚙️ Step 2: Imports & Configuration
This cell sets up the environment variables and imports the necessary modules.

**Key Actions:**
1.  **Silence Warnings:** Sets `TF_CPP_MIN_LOG_LEVEL` to '3' to hide aggressive TensorFlow logs that can clutter the output.
2.  **Import Libraries:**
    * `Flask`: For handling web requests.
    * `LangChain Community`: For connecting to Ollama and ChromaDB.
    * `LangChain Core`: For building the modern LCEL (LangChain Expression Language) pipeline.

In [2]:
import os
import warnings
import logging
import shutil
import sqlite3
import re
from datetime import datetime

# --- 1. CONFIGURATION (Silence Warnings) ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings("ignore")
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# --- 2. IMPORTS ---
from flask import Flask, render_template, request, jsonify, redirect, url_for

# AI & Database Components
from langchain_community.llms import Ollama
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Modern LCEL Components (The "Fix" for missing chains)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✅ Environment Configured & Libraries Imported.")


✅ Environment Configured & Libraries Imported.


### 🛡️ Step 3: Database & Safety Guardrails
This cell initializes the persistent storage and safety mechanisms.

**1. SQLite Database (`init_db`)**
* Creates a local file `therapist.db`.
* **`moods` table:** Stores user mood entries (Mood, Note, Date).
* **`chat_history` table:** Stores conversation logs to give the AI long-term memory.

**2. Safety Guardrails (`check_safety_guardrails`)**
* A function that scans user input for keywords like "suicide", "kill myself", or "overdose".
* Returns a hardcoded crisis message immediately if danger is detected, bypassing the AI.

In [3]:
# --- 1. PERSISTENT DATABASE SETUP ---
def init_db():
    """Creates the database tables if they don't exist."""
    conn = sqlite3.connect('therapist.db')
    c = conn.cursor()
    
    # Table 1: Mood Tracking
    c.execute('''
        CREATE TABLE IF NOT EXISTS moods (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            mood TEXT NOT NULL,
            note TEXT,
            date TEXT
        )
    ''')
    
    # Table 2: Chat History (For Persistent Memory)
    c.execute('''
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conn.commit()
    conn.close()

# Initialize the DB immediately
init_db()
print("✅ Database 'therapist.db' ready.")

def save_message(session_id, role, content):
    """Saves a message to the SQLite database."""
    try:
        conn = sqlite3.connect('therapist.db')
        c = conn.cursor()
        c.execute("INSERT INTO chat_history (session_id, role, content) VALUES (?, ?, ?)", 
                  (session_id, role, content))
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"Error saving message: {e}")

# --- 2. EMERGENCY GUARDRAILS ---
def check_safety_guardrails(user_text):
    """
    Scans input for crisis keywords. 
    Returns a crisis message if found, otherwise None.
    """
    danger_patterns = [
        r"\b(kill|hurt|harm) (myself|me)\b",
        r"\b(suicide|die|death)\b",
        r"\b(end) (my life)\b",
        r"\b(overdose)\b"
    ]
    
    # Check regex patterns
    for pattern in danger_patterns:
        if re.search(pattern, user_text, re.IGNORECASE):
            return (
                "⚠️ **EMERGENCY SAFETY ALERT** ⚠️\n\n"
                "I detected that you might be in immediate danger. I am an AI and cannot provide crisis intervention.\n\n"
                "Please perform one of the following actions immediately:\n"
                "- Call your local emergency number (e.g., 911, 999).\n"
                "- Call a Suicide Prevention Hotline.\n"
                "- Reach out to a trusted family member or friend.\n\n"
                "You are not alone, and help is available."
            )
    return None

print("✅ Safety Guardrails Active.")

✅ Database 'therapist.db' ready.
✅ Safety Guardrails Active.


### 🧠 Step 4: Build Knowledge Base (RAG)
This cell creates the "Brain" of the AI using Retrieval Augmented Generation (RAG).

**Process:**
1.  **Define Knowledge:** Writes specific therapy guidelines (e.g., 3-3-3 Rule, Box Breathing) to `knowledge.txt`.
2.  **Load & Split:** Uses `RecursiveCharacterTextSplitter` to break the text into manageable chunks (500 characters).
3.  **Embed:** Uses `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) to convert text into searchable vectors.
4.  **Store:** Saves these vectors into **ChromaDB** (a local folder named `./chroma_db`).

In [4]:
# --- 1. DEFINE KNOWLEDGE BASE ---
knowledge_content = """
GUIDELINES FOR AI THERAPIST:
1. ROLE: You are a compassionate, empathetic AI listener. You are NOT a doctor.
2. STRATEGY: Use "Active Listening". Validate feelings (e.g., "It sounds like you're carrying a heavy burden").
3. TECHNIQUE - ANXIETY (3-3-3 Rule): 
   - Ask the user to name 3 things they see, 3 sounds they hear, and move 3 parts of their body.
4. TECHNIQUE - PANIC (Box Breathing): 
   - Inhale for 4 seconds, Hold for 4 seconds, Exhale for 4 seconds, Hold for 4 seconds.
5. TECHNIQUE - DEPRESSION: 
   - Behavioral Activation: Encourage doing one tiny task (e.g., "Just wash one cup").
   - Social Connection: Suggest texting a friend just to say "hi".
6. SAFETY PROTOCOL: 
   - If the user mentions suicide, self-harm, or killing themselves, you MUST say: 
     "I am an AI and cannot provide crisis support. Please contact a local emergency number or suicide hotline immediately."
"""

# Write to file
with open("knowledge.txt", "w") as f:
    f.write(knowledge_content)

# --- 2. BUILD VECTOR DATABASE ---
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

loader = TextLoader("knowledge.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print("⏳ Generating Embeddings...")
# Using HuggingFace Embeddings (CPU optimized)
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

print("✅ Vector Database built successfully.")

⏳ Generating Embeddings...
✅ Vector Database built successfully.


### 🔗 Step 5: Build AI Pipeline
This cell constructs the "Cognitive Chain" that processes user queries.

**Components:**
1.  **Retriever:** Configured to fetch the top 3 most relevant chunks from ChromaDB.
2.  **LLM:** Connects to the local **Ollama** instance running `llama3.2:1b`.
3.  **Prompt Template:** A specific instruction set that tells the AI to act as a therapist and use the retrieved Context + History.
4.  **LCEL Chain:** Uses the pipe (`|`) syntax to create a workflow:
    * *Input* → *Retrieve Context* → *Format Prompt* → *Generate Answer* → *Parse String*.

In [5]:
print("⚙️ Building AI Pipeline...")

# 1. SETUP COMPONENTS
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
llm = Ollama(model="llama3.2:1b", temperature=0.6)

# 2. DEFINE PROMPT WITH HISTORY SUPPORT
# We explicitly add a {history} placeholder
# --- REVISED SYSTEM PROMPT (Action-Oriented) ---
template = """You are a pro-active AI Therapist Assistant. 
Your goal is to provide actionable mental health coping strategies immediately, not just to listen.

INSTRUCTIONS:
1. VALIDATE: Briefly acknowledge the user's pain in 1 short sentence (e.g., "I'm sorry you're hurting.").
2. DIAGNOSE (Informal): Identify if the user mentions Anxiety, Panic, or Depression based on the Context.
3. PRESCRIBE: Immediately provide the specific technique from the Context below (e.g., Box Breathing, 3-3-3 Rule, Behavioral Activation). 
   - Do NOT ask follow-up questions like "Can you tell me more?" if a technique is available.
   - Do NOT say "I cannot diagnose." Just say "Here is a technique that might help."
4. STYLE: Be warm but direct. Stop asking questions; start giving answers.

Chat History:
{history}

Context (Therapy Techniques):
{context}

User's Input: {input}

Therapist Response:
"""

prompt = ChatPromptTemplate.from_template(template)

# 3. BUILD CHAIN (LCEL)
# We format the docs into a string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# The Pipeline: 
# 1. Retrieve Context
# 2. Pass Context + History + Input to Prompt
# 3. Pass to LLM -> String Parser
rag_chain = (
    {
        "context": retriever | format_docs, 
        "history": RunnablePassthrough(), # We will pass history string manually
        "input": RunnablePassthrough()    # We will pass input string manually
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ AI Chain Ready (Manual Memory Architecture).")

⚙️ Building AI Pipeline...
✅ AI Chain Ready (Manual Memory Architecture).


C:\Users\faraz\AppData\Local\Temp\ipykernel_14644\2808392286.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.2:1b", temperature=0.6)


### 🚀 Step 6: Flask Server Application
This is the main application logic that runs the web server.

**Routes:**
* **`/` (GET):** Renders the Mood Tracker UI and fetches past moods from SQLite.
* **`/add_mood` (POST):** Accepts form data and saves new mood entries to SQLite.
* **`/chat` (GET):** Renders the Chatbot Interface.
* **`/chat` (POST):** The core logic hub:
    1.  **Contextualizes Input:** Appends Age, Sex, and Duration to the user's message as metadata.
    2.  **Safety Check:** Runs the guardrail function before calling the AI.
    3.  **AI Generation:** Invokes the RAG chain defined in Step 5.
    4.  **Memory:** Saves both User and AI messages to the `chat_history` table.
    5.  **Response:** Returns the clean text to the frontend.

In [6]:
# --- FLASK APP ---
app = Flask(__name__)

@app.route("/")
def index():
    conn = sqlite3.connect('therapist.db')
    conn.row_factory = sqlite3.Row
    c = conn.cursor()
    try:
        c.execute("SELECT * FROM moods ORDER BY id DESC")
        moods = c.fetchall()
    except: moods = []
    conn.close()
    return render_template("mood.html", moods=moods)

@app.route("/add_mood", methods=["POST"]) # type: ignore
def add_mood():
    if request.method == "POST":
        mood = request.form.get("mood")
        note = request.form.get("note")
        if mood:
            conn = sqlite3.connect('therapist.db')
            c = conn.cursor()
            c.execute("INSERT INTO moods (mood, note, date) VALUES (?, ?, ?)", 
                      (mood, note, datetime.now().strftime("%Y-%m-%d %H:%M")))
            conn.commit()
            conn.close()
        return redirect(url_for("index"))

@app.route("/chat")
def chatbot():
    return render_template("chatbot.html")

@app.route("/chat", methods=["POST"])
def chat():
    try:
        data = request.get_json()
        
        # Extract inputs
        user_message = data.get("symptoms", "") 
        duration = data.get("duration", "")
        age = data.get("age", "")
        sex = data.get("sex", "")
        session_id = data.get("session_id", "guest")

        # --- CONTEXTUAL PROMPT FIX ---
        # The AI sees the User Message first. The details are added as separate context.
        # This prevents the "I feel [Question]" grammar error.
        user_input = f"{user_message} (Context: Client is {age} years old {sex}, duration: {duration})"

        # 1. Safety Guardrail
        if re.search(r"\b(kill|suicide|die|hurt)\b", user_input, re.IGNORECASE):
            return jsonify({"response": "⚠️ **Safety Alert:** Please contact emergency services immediately."})

        # 2. Run AI
        # (Assumes rag_chain is defined in an intermediate cell)
        response_data = rag_chain.invoke(user_input)

        # 3. Extract Response (Handle Dict vs String)
        if isinstance(response_data, dict):
            bot_text = response_data.get("answer", response_data.get("result", str(response_data)))
        else:
            bot_text = str(response_data)
            
        # 4. Clean Formatting
        clean_response = bot_text.replace("*", "").strip()

        # 5. Save to Memory (Using the helper from Part 1)
        save_message(session_id, "User", user_message)
        save_message(session_id, "Therapist", clean_response)

        return jsonify({"response": clean_response})

    except Exception as e:
        print(f"❌ Error: {e}")
        return jsonify({"error": "I am having trouble connecting to my memory."}), 500

if __name__ == "__main__":
    from werkzeug.serving import run_simple
    print("🚀 Server Starting (Fixed Logic)...")
    run_simple('localhost', 5000, app)

 * Running on http://localhost:5000
Press CTRL+C to quit


🚀 Server Starting (Fixed Logic)...


127.0.0.1 - - [14/Jan/2026 13:48:51] "GET /chat HTTP/1.1" 200 -
127.0.0.1 - - [14/Jan/2026 13:48:52] "GET /static/compiled-tailwind.css HTTP/1.1" 304 -
127.0.0.1 - - [14/Jan/2026 13:48:52] "GET /static/script.js HTTP/1.1" 304 -
